In [1]:
!pip install pycuda
#negociodemoradodocarai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 7.1 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659447 sha256=3945c59bf0edd0e752c0c047e984c76c27cbb107e708ddeaeb0ac5992f21f0d7
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [2]:
from numba import cuda

In [3]:
import os, time
import numpy as np

import pycuda.driver as drv
import pycuda.autoinit
from pycuda.compiler import SourceModule

from cryptography.hazmat.primitives.ciphers import Cipher, algorithms

dev = drv.Device(0)
print("GPU do GoogleColab: ", dev.name())
print("Memoria total: ", dev.total_memory() / 1e9, "GB")
print("Compute capability?: ", dev.compute_capability())

GPU do GoogleColab:  Tesla T4
Memoria total:  15.637086208 GB
Compute capability?:  (7, 5)


In [4]:
mod = SourceModule("""
__global__ void dobrar(int *a) {
    int indiceGlobal = blockIdx.x * blockDim.x + threadIdx.x;
    a[indiceGlobal] *= 2;
}
""")
dobrar = mod.get_function("dobrar")

In [5]:

a = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=np.int32)
print("antes: ", a)
dobrar(drv.InOut(a), grid=(2,1,1), block=(4,1,1))
print("depois: ", a)



antes:  [1 2 3 4 5 6 7 8]
depois:  [ 2  4  6  8 10 12 14 16]


In [6]:
mod2 = SourceModule("""
__global__ void rotular(int *blocos, int *threads) {
    int indiceGlobal = blockIdx.x * blockDim.x + threadIdx.x;
    blocos[indiceGlobal]  = blockIdx.x;
    threads[indiceGlobal] = threadIdx.x;
}
""")
rotular = mod2.get_function("rotular")

b = np.zeros(8, dtype=np.int32)
t = np.zeros(8, dtype=np.int32)
rotular(drv.Out(b), drv.Out(t), grid=(2,1,1), block=(4,1,1))

print("posição: ", list(range(8)))
print("bloco: ", b)
print("thread :", t)

posição:  [0, 1, 2, 3, 4, 5, 6, 7]
bloco:  [0 0 0 0 1 1 1 1]
thread : [0 1 2 3 0 1 2 3]


In [7]:
# XOR mais simples que ChaCha, testes.




In [8]:

mod = SourceModule("""
__global__ void xor_buffer(unsigned char *buf, unsigned char chave, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < n) {
        buf[tid] ^= chave;
    }
}
""")
xor_gpu = mod.get_function("xor_buffer")

mensagem = bytearray(b"Rafael Sampaio e Silva")
buf = np.frombuffer(mensagem, dtype=np.uint8).copy()
chave = np.uint8(0x42)

threads = 32
blocos = (len(buf) + threads - 1) // threads

xor_gpu(drv.InOut(buf), chave, np.int32(len(buf)),
        grid=(blocos,1,1), block=(threads,1,1))
print("cifrado :", bytes(buf))

xor_gpu(drv.InOut(buf), chave, np.int32(len(buf)),
        grid=(blocos,1,1), block=(threads,1,1))
print("decifrado:", bytes(buf))

cifrado : b"\x10#$#'.b\x11#/2#+-b'b\x11+.4#"
decifrado: b'Rafael Sampaio e Silva'


In [1]:
"""
ChaCha20 paralelo em GPU com PyCUDA.

Conceitos de Programação Paralela aplicados:
  - SourceModule  -> kernel CUDA em C, compilado em runtime
  - __global__    -> função que roda na GPU, chamada da CPU
  - blockIdx.x, threadIdx.x, blockDim.x -> índice global da thread
  - drv.Event()   -> cronometragem precisa de GPU (assíncrona)
  - assert        -> verificação de correção contra a biblioteca
"""

import os, time
import numpy as np
import pycuda.driver as drv
import pycuda.autoinit
from pycuda.compiler import SourceModule

from cryptography.hazmat.primitives.ciphers import Cipher, algorithms

# =====================================================================
# 1) KERNEL CUDA — cada thread cifra um bloco de 64 bytes
# =====================================================================
mod = SourceModule(r"""
#include <stdint.h>

#define ROTL32(v, n) (((v) << (n)) | ((v) >> (32 - (n))))

#define QR(a, b, c, d) {                          \
    a += b; d = ROTL32(d ^ a, 16);                \
    c += d; b = ROTL32(b ^ c, 12);                \
    a += b; d = ROTL32(d ^ a,  8);                \
    c += d; b = ROTL32(b ^ c,  7);                \
}

__global__ void chacha20(
    const uint32_t *key,     // 8 palavras (32 bytes)
    const uint32_t *nonce,   // 3 palavras (12 bytes)
    const uint8_t  *pt,
    uint8_t        *ct,
    int             n_blocks)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= n_blocks) return;

    // monta o estado: 4 constantes + 8 key + 1 counter + 3 nonce
    uint32_t s[16];
    s[0]=0x61707865; s[1]=0x3320646e; s[2]=0x79622d32; s[3]=0x6b206574;
    for (int i=0; i<8; i++) s[4+i]  = key[i];
    s[12] = (uint32_t)tid;                  // contador = id da thread
    for (int i=0; i<3; i++) s[13+i] = nonce[i];

    uint32_t w[16];
    for (int i=0; i<16; i++) w[i] = s[i];

    // 20 rounds = 10 (col) + 10 (diag)
    for (int r=0; r<10; r++) {
        QR(w[0], w[4], w[ 8], w[12]);
        QR(w[1], w[5], w[ 9], w[13]);
        QR(w[2], w[6], w[10], w[14]);
        QR(w[3], w[7], w[11], w[15]);
        QR(w[0], w[5], w[10], w[15]);
        QR(w[1], w[6], w[11], w[12]);
        QR(w[2], w[7], w[ 8], w[13]);
        QR(w[3], w[4], w[ 9], w[14]);
    }
    for (int i=0; i<16; i++) w[i] += s[i];

    // XOR keystream com plaintext (64 bytes da minha thread)
    int base = tid * 64;
    for (int i=0; i<16; i++) {
        ct[base+i*4+0] = pt[base+i*4+0] ^ ( w[i]        & 0xFF);
        ct[base+i*4+1] = pt[base+i*4+1] ^ ((w[i] >>  8) & 0xFF);
        ct[base+i*4+2] = pt[base+i*4+2] ^ ((w[i] >> 16) & 0xFF);
        ct[base+i*4+3] = pt[base+i*4+3] ^ ((w[i] >> 24) & 0xFF);
    }
}
""")
chacha20_gpu = mod.get_function("chacha20")

# ======================================================
# 2) DADOS
# ========================================
N_BLOCKS = 1_000_000 # 1M blocos × 64 B = 64 MB
N_BYTES  = N_BLOCKS * 64

key_bytes   = bytes(32) # chave demo: 32 zeros (não use em produção!)
nonce_bytes = bytes(12) # nonce demo: 12 zeros
plaintext   = os.urandom(N_BYTES)

key_u32   = np.frombuffer(key_bytes,   dtype=np.uint32)
nonce_u32 = np.frombuffer(nonce_bytes, dtype=np.uint32)
pt_np     = np.frombuffer(plaintext,   dtype=np.uint8).copy()
ct_np     = np.empty_like(pt_np)

# =======================================================
# 3) WARM-UP + GPU TIMING (estilo aula: drv.Event)
# ======================================================
threads_per_block = 256
n_grid = (N_BLOCKS + threads_per_block - 1) // threads_per_block

# warm-up para não cronometrar a 1ª chamada (overhead de inicialização)
chacha20_gpu(
    drv.In(key_u32), drv.In(nonce_u32),
    drv.In(pt_np[:64*100].copy()), drv.Out(ct_np[:64*100]),
    np.int32(100),
    grid=(1,1,1), block=(threads_per_block,1,1),
)

start, end = drv.Event(), drv.Event()
start.record()
chacha20_gpu(
    drv.In(key_u32), drv.In(nonce_u32),
    drv.In(pt_np), drv.Out(ct_np),
    np.int32(N_BLOCKS),
    grid=(n_grid, 1, 1),
    block=(threads_per_block, 1, 1),
)
end.record(); end.synchronize()
gpu_ms = start.time_till(end)
print(f"GPU  : {gpu_ms:8.2f} ms  ({N_BYTES/1e6:.1f} MB, {n_grid} blocos × {threads_per_block} threads)")

# =====================================================================
# 4) BASELINE CPU (biblioteca otimizada da aula = referência absoluta)
# =====================================================================
# A 'cryptography' usa nonce de 16 bytes = counter(4 LE) || nonce(12)
nonce_lib = (0).to_bytes(4, "little") + nonce_bytes
t0 = time.perf_counter()
enc = Cipher(algorithms.ChaCha20(key_bytes, nonce_lib), mode=None).encryptor()
ct_cpu = enc.update(plaintext)
cpu_ms = (time.perf_counter() - t0) * 1000
print(f"CPU  : {cpu_ms:8.2f} ms")

# =====================================================================
# 5) VERIFICAÇÃO DE CORREÇÃO
# =====================================================================
assert bytes(ct_np) == ct_cpu, "GPU e CPU divergiram!"
print(f"OK   : ciphertexts batem em {N_BYTES/1e6:.1f} MB.")
print(f"Speedup vs CPU (lib): {cpu_ms / gpu_ms:.2f}x")

ModuleNotFoundError: No module named 'pycuda'